# Colab Experiment Runner Template
Use this notebook to run experiments on Google Colab. 
It automatically handles repository updates, environment setups, high-speed dataset copying to local SSD, and runs outputs synchronization to Google Drive.

In [ ]:
# ==============================================================================
# Cell 0: Setup Environment & Transfer Dataset
# ==============================================================================
import os
import shutil
import subprocess

# --- CONFIGURATION ---
DATASET = "vaihingen"      # Options: vaihingen, potsdamRGB, loveda
REPO_URL = "https://github.com/hoangnecon/cdga-experiments.git"
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_ROOT = f"{DRIVE_MOUNT_POINT}/MyDrive/WACV_EXP"
# ---------------------

# 1. Mount Google Drive
if not os.path.exists(DRIVE_MOUNT_POINT):
    print("[1/4] Mounting Google Drive...")
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
else:
    print("[1/4] Google Drive already mounted.")

# 2. Clone/Update Code Repository
PROJECT_DIR = "/content/cdga-experiments"
if not os.path.exists(PROJECT_DIR):
    print(f"[2/4] Cloning code repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"[2/4] Pulling latest code in {PROJECT_DIR}...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

# 3. Setup Python environment and clone dependencies
print("[3/4] Running setup_colab.sh...")
subprocess.run(["bash", f"{PROJECT_DIR}/setup_colab.sh"], check=True)

# 4. Transfer dataset to local VM SSD for high-speed I/O
print(f"[4/4] Preparing dataset '{DATASET}' on local SSD...")
LOCAL_DATA_DIR = f"/content/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Path to the dataset on Google Drive
drive_dataset_dir = f"{DRIVE_ROOT}/data/{DATASET}"
drive_dataset_tar = f"{DRIVE_ROOT}/data/{DATASET}.tar"
local_dataset_dest = f"{LOCAL_DATA_DIR}/{DATASET}"

if os.path.exists(local_dataset_dest):
    print(f"  ✓ Dataset already exists at {local_dataset_dest}.")
else:
    # Check if zipped/tarred version exists on Drive (recommended for speed)
    if os.path.exists(drive_dataset_tar):
        print(f"  Found tar archive on Drive: {drive_dataset_tar}")
        print("  Extracting archive directly to local SSD (this is fast)...")
        subprocess.run(["tar", "-xf", drive_dataset_tar, "-C", LOCAL_DATA_DIR], check=True)
        print("  ✓ Extraction complete.")
    elif os.path.exists(drive_dataset_dir):
        print(f"  Tar archive not found. Copying directory from Drive (this may take time)...")
        print(f"  Copying {drive_dataset_dir} to {local_dataset_dest}...")
        shutil.copytree(drive_dataset_dir, local_dataset_dest)
        print("  ✓ Copy complete.")
        # Auto-create tar file on Drive for next time to speed up startup!
        print(f"  Auto-archiving dataset to {drive_dataset_tar} for faster startup next time...")
        try:
            subprocess.run(["tar", "-cf", drive_dataset_tar, "-C", LOCAL_DATA_DIR, DATASET], check=True)
            print("  ✓ Tar archive created successfully on Google Drive.")
        except Exception as e:
            print(f"  [Warning] Failed to archive dataset: {e}")
    else:
        raise FileNotFoundError(f"Dataset not found on Google Drive: searched for '{drive_dataset_tar}' and '{drive_dataset_dir}'")

# Export environment paths
os.environ["PYTHONPATH"] = f"{PROJECT_DIR}:{PROJECT_DIR}/geoseg:" + os.environ.get("PYTHONPATH", "")
os.environ["GEOSEG_DIR"] = f"{PROJECT_DIR}/geoseg"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("\n=== Setup completed successfully! Ready for verification. ===")

In [ ]:
# ==============================================================================
# Cell 1: Verify Environment & Data Loading
# ==============================================================================
import os
import sys
import torch
from pathlib import Path

# Config
DATASET = "vaihingen"  # Options: vaihingen, potsdamRGB, loveda
METHOD = "cdga"        # Options: cdga, baseline_ce, baseline_abl, etc.
CONFIG_FILE = "experiments/configs/methods/gradient_based/cdga.yaml"

PROJECT_DIR = "/content/cdga-experiments"
LOCAL_DATA_DIR = f"/content/data/{DATASET}"

# Ensure paths are added
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if f"{PROJECT_DIR}/geoseg" not in sys.path:
    sys.path.insert(0, f"{PROJECT_DIR}/geoseg")

# Import dataset and trainer
from shared.trainer import load_config
from shared.trainer import get_dataset_class

print("=== [1/3] Verifying Dataset Loader ===")
dataset_cls = get_dataset_class(DATASET)
ds = dataset_cls(split="train", data_root=LOCAL_DATA_DIR)
print(f"Successfully loaded dataset: {DATASET} (Total train samples: {len(ds)})")

# Label distribution check
print("Checking label distribution (first 10 samples)...")
label_counts = {}
for i in range(min(10, len(ds))):
    lbl = ds[i]["label"]
    for u, c in zip(*torch.unique(lbl, return_counts=True)):
        label_counts[int(u)] = label_counts.get(int(u), 0) + int(c)
for k in sorted(label_counts.keys()):
    print(f"  class index {k}: {label_counts[k]:>10,}")

print("\n=== [2/3] Verifying Config Loading ===")
config_path = Path(f"{PROJECT_DIR}/{CONFIG_FILE}")
cfg = load_config(config_path)
print("Config loaded successfully:")
print(f"  Dataset: {cfg['data']['dataset']}")
print(f"  Num Classes: {cfg['data']['num_classes']}")
print(f"  Ignore Index: {cfg['data']['ignore_index']}")
print(f"  Backbone: {cfg['model']['backbone']}")

print("\n=== [3/3] Performing Model Forward/Backward Test ===")
import importlib.util

# Resolve build_model from method directory
method_dir_slug = "gradient_based" if "gradient" in CONFIG_FILE else "loss_based"
model_module_path = f"{PROJECT_DIR}/experiments/methods/group_a/{method_dir_slug}/{METHOD}/model.py"

spec = importlib.util.spec_from_file_location("model_module", model_module_path)
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)

model = model_module.build_model(cfg)
model.backbone = model.backbone.cuda()
model.train_mode()

sample = ds[0]
image = sample["image"].unsqueeze(0).cuda()
label = sample["label"].unsqueeze(0).cuda()
boundary_mask = sample.get("boundary_mask")
if boundary_mask is not None:
    boundary_mask = boundary_mask.unsqueeze(0).cuda()

loss = model.forward_train(image, label, boundary_mask=boundary_mask)
loss.backward()
print(f"✓ Forward & backward pass completed. Loss: {loss.item():.4f}")

# Check hook grad stats if CDGA
if hasattr(model, "_hook") and model._hook.grad_stats:
    stats = model._hook.grad_stats
    r = stats.get('mod_boundary', 0) / max(stats.get('orig_boundary', 1e-8), 1e-8)
    print(f"✓ CDGA Gradient Amplification Ratio: {r:.1f}x")

print("\n>>> ALL CHECKS PASSED — READY TO TRAIN! <<<")

# Run Experiment

In [ ]:
# ==============================================================================
# Cell 2: Run Training
# ==============================================================================
import os

# --- RUN PARAMETERS ---
DATASET = "vaihingen"      # Options: vaihingen, potsdamRGB, loveda
METHOD = "cdga"            # Options: cdga, baseline_ce, baseline_abl, etc.
CONFIG_NAME = "cdga.yaml"  # Config file name
BATCH_SIZE = 8             # Batch size override (decrease if out of memory)
EPOCHS = 100               # Number of epochs override
# ----------------------

PROJECT_DIR = "/content/cdga-experiments"
LOCAL_DATA_DIR = f"/content/data/{DATASET}"
TIER = "gradient_based" if METHOD in ["cdga", "pcgrad"] else "loss_based"

TRAIN_SCRIPT = f"/content/cdga-experiments/experiments/methods/group_a/{TIER}/{METHOD}/train.py"
CONFIG_PATH = f"/content/cdga-experiments/experiments/configs/methods/{TIER}/{CONFIG_NAME}"
ENV_CONFIG = f"/content/cdga-experiments/experiments/configs/colab_env.yaml"

assert os.path.exists(TRAIN_SCRIPT), f"Training script not found: {TRAIN_SCRIPT}"
assert os.path.exists(CONFIG_PATH), f"Config file not found: {CONFIG_PATH}"
assert os.path.exists(ENV_CONFIG), f"Environment config not found: {ENV_CONFIG}"

# Build command list
cmd = f"""python "{TRAIN_SCRIPT}" \
    --config "{CONFIG_PATH}" \
    --env-config "{ENV_CONFIG}" \
    --override \
        data.data_root="{LOCAL_DATA_DIR}" \
        train.batch_size={BATCH_SIZE} \
        train.epochs={EPOCHS}
"""

print("Running command:")
print(cmd)
print("\n" + "="*80 + "\n")

# Run in shell
!cd "{PROJECT_DIR}" && {cmd}